# Week 2: Structured Output - 추출 텍스트를 JSON으로 구조화

지난주에 이미지에서 텍스트를 추출했습니다. 이번 주에는 그 텍스트에서
**환자명, 진단명, 처방 등 필요한 필드만 뽑아서 정형 데이터(JSON)로 변환**합니다.

## 파이프라인
```
EMR 이미지 → GPT-5.4-nano (텍스트 추출) → GPT-5.4-nano (JSON 구조화) → 정형 데이터
```

## 핵심 개념
- **Structured Output**: LLM이 자유로운 텍스트 대신 정해진 형식(JSON)으로 답하게 하는 기법
- **response_format**: OpenAI API에서 JSON 출력을 강제하는 파라미터

---

## 1. 환경 설정

In [ ]:
!pip install -q openai pillow

import os
from getpass import getpass

api_key = getpass("OpenAI API Key를 입력하세요: ")
os.environ["OPENAI_API_KEY"] = api_key
print("설정 완료!")

## 2. 지난주 복습 - 이미지에서 텍스트 추출

Week 1에서 만든 함수를 그대로 사용합니다.

In [ ]:
import base64
from openai import OpenAI

client = OpenAI()


def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def extract_text_from_image(image_path):
    """이미지에서 텍스트를 추출한다."""
    base64_image = encode_image(image_path)
    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "이 의료 문서 이미지에서 보이는 모든 텍스트를 정확하게 추출해주세요. "
                           "문서의 레이아웃을 최대한 유지하면서 추출해주세요."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{base64_image}",
                        "detail": "high"
                    }
                }
            ]
        }],
        max_completion_tokens=4096,
    )
    return response.choices[0].message.content


print("함수 준비 완료!")

## 3. 샘플 데이터 준비 & 텍스트 추출

In [ ]:
import glob

# 샘플 이미지 준비 (Week 1과 동일)
!git clone --depth 1 https://github.com/Yuri-daepo/hospital-emr-ocr.git /content/repo 2>/dev/null || true
!cp /content/repo/emr_samples/*.png /content/ 2>/dev/null || true

samples = sorted(glob.glob("/content/*.png"))
print(f"샘플 {len(samples)}개 준비 완료")

# 첫 번째 이미지에서 텍스트 추출
img_path = samples[0]
print(f"\n텍스트 추출 중: {os.path.basename(img_path)}...")
raw_text = extract_text_from_image(img_path)

print(f"\n--- 추출된 텍스트 ---")
print(raw_text)

## 4. Structured Output - JSON으로 구조화 (핵심!)

추출된 비정형 텍스트를 LLM에 다시 보내서 정해진 필드의 JSON으로 변환합니다.

### 왜 2단계로 나누나?
- 1단계 (이미지→텍스트): 이미지 인식에 집중
- 2단계 (텍스트→JSON): 데이터 구조화에 집중
- 각 단계가 한 가지에 집중하면 정확도가 올라감

In [ ]:
import json

STRUCTURING_PROMPT = """당신은 의료 문서 데이터 추출 전문가입니다.

아래는 병원 EMR 스캔본에서 추출한 텍스트입니다.
이 텍스트에서 아래 필드를 추출하여 JSON으로 구조화하세요.

## 추출할 필드
- hospital_name: 병원명
- patient_name: 환자명
- patient_id: 환자번호/등록번호
- birth_date: 생년월일 (YYYY-MM-DD)
- gender: 성별 (M/F)
- visit_date: 진료일/내원일 (YYYY-MM-DD)
- department: 진료과
- doctor_name: 담당의/주치의
- diagnosis: 진단명 (배열)
- chief_complaint: 주소/주증상
- vital_signs: 활력징후 (객체: BP, HR, BT 등)
- medications: 처방 약물 (배열, 각 항목에 drug_name, dose, frequency)
- notes: 기타 소견/메모

## 규칙
1. 문서에 없는 필드는 null로 표시
2. OCR 오류로 보이는 오탈자는 교정
3. 날짜는 YYYY-MM-DD 형식으로 통일
4. 반드시 유효한 JSON만 출력 (설명 텍스트 없이)

## 추출 대상 텍스트
{text}
"""


def structure_emr_text(raw_text):
    """비정형 텍스트를 구조화된 JSON으로 변환한다."""
    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[{
            "role": "user",
            "content": STRUCTURING_PROMPT.format(text=raw_text)
        }],
        temperature=0,
        response_format={"type": "json_object"},  # ← 핵심! JSON 출력 강제
    )
    return json.loads(response.choices[0].message.content)


print("함수 정의 완료!")

## 5. 구조화 실행

In [ ]:
print("구조화 중...\n")
structured = structure_emr_text(raw_text)

print(json.dumps(structured, ensure_ascii=False, indent=2))

## 6. 여러 이미지에 대해 일괄 구조화

In [ ]:
all_results = {}

for img_path in samples:
    name = os.path.basename(img_path)
    print(f"처리 중: {name}")

    try:
        # 1단계: 이미지 → 텍스트
        print(f"  1) 텍스트 추출...")
        text = extract_text_from_image(img_path)

        # 2단계: 텍스트 → JSON
        print(f"  2) JSON 구조화...")
        data = structure_emr_text(text)

        all_results[name] = data
        print(f"  → 완료\n")

    except Exception as e:
        print(f"  → 실패: {e}\n")
        all_results[name] = {"error": str(e)}

print(f"=== 전체 {len(all_results)}개 처리 완료 ===")

## 7. 결과 비교 - 병원별로 어떻게 다른지 확인

In [ ]:
for name, data in all_results.items():
    print(f"\n{'=' * 60}")
    print(f"파일: {name}")
    print(f"{'=' * 60}")
    print(json.dumps(data, ensure_ascii=False, indent=2))

## 8. (실습) 추출 필드 커스터마이징

프롬프트의 "추출할 필드"를 수정하여 원하는 데이터만 뽑아보세요.

**시도해볼 것:**
- 약물 정보만 뽑기
- KCD 진단코드 추정 추가
- 보험 청구에 필요한 필드만 추출

In [ ]:
# === 프롬프트를 자유롭게 수정해보세요 ===
MY_PROMPT = """이 EMR 텍스트에서 건강보험 청구에 필요한 정보만 추출하세요.

추출할 필드:
- patient_name: 환자명
- patient_id: 환자번호
- visit_date: 진료일
- diagnosis_with_code: 진단명 및 추정 KCD코드 (배열)
- procedures: 시행한 검사/처치 (배열)
- medications: 처방약 (배열)

JSON만 출력하세요.

텍스트:
{text}
"""
# ========================================

response = client.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[{"role": "user", "content": MY_PROMPT.format(text=raw_text)}],
    temperature=0,
    response_format={"type": "json_object"},
)

custom_result = json.loads(response.choices[0].message.content)
print(json.dumps(custom_result, ensure_ascii=False, indent=2))

---

## 정리

### 이번 주에 배운 것
- `response_format={"type": "json_object"}`로 LLM 출력을 JSON으로 강제
- 프롬프트에 추출할 필드를 명시하면 원하는 구조로 데이터 추출 가능
- 병원마다 양식이 달라도 LLM이 맥락을 파악하여 동일한 필드로 통일

### 핵심 포인트
- **프롬프트가 곧 스키마**: 코드를 바꾸지 않고 프롬프트만 수정하면 추출 필드를 변경 가능
- **2단계 분리**: 이미지→텍스트, 텍스트→JSON을 나누면 각 단계의 정확도가 향상
- **response_format**: 항상 유효한 JSON을 보장 (파싱 에러 방지)

### 다음 주 예고
이번 주에 만든 기능을 **하나의 파이프라인으로 연결**하고,
여러 파일을 일괄 처리하여 **CSV/Excel로 내보내는** 자동화를 만듭니다.